# DataFrame Modifications using withColumn Method

This notebook demonstrates various DataFrame transformation techniques using the `withColumn()` method.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, lit, when

spark = SparkSession.builder \
    .appName('WithColumn Examples') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Create Sample DataFrame

In [ ]:
# Sample data
data1 = [
    ('James', '', 'Smith', '1991-04-01', 'M', 3000),
    ('Michael', 'Rose', '', '2000-05-19', 'M', 4000),
    ('Robert', '', 'Williams', '1978-09-05', 'M', 4000),
    ('Maria', 'Anne', 'Jones', '1967-12-01', 'F', 4000),
    ('Jen', 'Mary', 'Brown', '1980-02-17', 'F', -1),
    ('JM ABCDEFGHIJKLMNOPQRSTUVWXYZ', '', '', '1982-04-17', 'M', 5000)
]

columns = ["firstname", "middlename", "lastname", "dob", "gender", "salary"]

df = spark.createDataFrame(data=data1, schema=columns)

print("Original DataFrame:")
df.printSchema()
df.show(truncate=False)

## 1. Change Column Data Type

In [ ]:
# Cast salary to Integer type
df2 = df.withColumn("salary2", col("salary").cast("Integer"))

print("After adding salary2 with Integer type:")
df2.printSchema()
df2.show(truncate=False)

## 2. Update Existing Column Value

In [ ]:
# Multiply salary by 100
df3 = df.withColumn("salary", col("salary") * 100)

print("After multiplying salary by 100:")
df3.printSchema()
df3.show(truncate=False)

## 3. Conditional Column using when()

In [ ]:
# Create age_gender column based on condition
dfn = df.withColumn(
    "age_gender", 
    when(col("gender") == 'M', "Male").otherwise("Female")
)

print("With conditional age_gender column:")
dfn.show(truncate=False)

## 4. Add Derived Column

In [ ]:
# Add new column derived from existing column
df4 = df.withColumn("NewSalaryColumn", col("salary") * -1)

print("With derived NewSalaryColumn:")
df4.printSchema()
df4.show(truncate=False)

## 5. Add Column with Constant Value

In [ ]:
# Add constant value column
df5 = df.withColumn("Country", lit("USA"))

print("With constant Country column:")
df5.printSchema()
df5.show(truncate=False)

## 6. Add Multiple Columns

In [ ]:
# Chain multiple withColumn operations
df6 = df \
    .withColumn("Country", lit("USA")) \
    .withColumn("anotherColumn", lit("anotherValue"))

print("With multiple new columns:")
df6.printSchema()
df6.show(truncate=False)

## 7. Rename Column

In [ ]:
# Rename firstname to fname
df_renamed = df.withColumnRenamed("firstname", "fname")

print("After renaming firstname to fname:")
df_renamed.show(truncate=False)

## 8. Drop Column

In [ ]:
# Drop NewSalaryColumn
df_dropped = df4.drop("NewSalaryColumn")

print("After dropping NewSalaryColumn:")
df_dropped.show(truncate=False)

## 9. Complex Transformations

In [ ]:
# Multiple transformations at once
df_complex = df \
    .withColumn("salary_doubled", col("salary") * 2) \
    .withColumn("salary_category", 
                when(col("salary") < 3000, "Low")
                .when(col("salary") < 4500, "Medium")
                .otherwise("High")) \
    .withColumn("gender_full", 
                when(col("gender") == "M", "Male")
                .otherwise("Female")) \
    .withColumn("country", lit("USA"))

print("Complex transformations:")
df_complex.show(truncate=False)

## 10. Working with Date Columns

In [ ]:
from pyspark.sql.functions import to_date, year, month, datediff, current_date

# Convert string to date and extract date components
df_dates = df \
    .withColumn("dob_date", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("birth_year", year(col("dob_date"))) \
    .withColumn("birth_month", month(col("dob_date"))) \
    .withColumn("days_since_birth", datediff(current_date(), col("dob_date")))

print("Date transformations:")
df_dates.select("firstname", "dob", "birth_year", "birth_month", "days_since_birth").show()

In [ ]:
# Stop Spark Session
spark.stop()